# The Nyquist criterion and stability margins: notebook

The running example of the Note: the robot's heading loop. Yaw inertia I = 0.1 kg m², PD gains Kp = 0.4 N m/rad and Kd = 0.08 N m s/rad (the derivative acts on the measured heading), torque limit 0.2 N m, and a delay τ between the heading and the controller. We check every number in the Note.

In [1]:
import numpy as np
from scipy.optimize import brentq, fsolve
from scipy.signal import find_peaks

I, KP, KD, TAU_MAX = 0.1, 0.4, 0.08, 0.2


def L(s, tau=0.0, k=1.0, kd=KD):
    """Loop transfer function C(s) G(s) e^{-s tau} = k (Kp + Kd s) / (I s^2) e^{-s tau}."""
    return k * (KP + kd * s) / (I * s**2) * np.exp(-s * tau)


def simulate(t, ref, tau=0.0):
    """I theta'' = torque, torque = Kp (ref - theta) - Kd theta', both seen tau s late, clipped at 0.2 N m."""
    dt, lag = t[1] - t[0], int(round(tau / (t[1] - t[0])))
    th, om, tq = np.zeros(len(t)), np.zeros(len(t)), np.zeros(len(t))
    for i in range(len(t) - 1):
        j = max(i - lag, 0)
        tq[i] = np.clip(KP * (ref - th[j]) - KD * om[j], -TAU_MAX, TAU_MAX)
        om[i + 1] = om[i] + dt * tq[i] / I
        th[i + 1] = th[i] + dt * om[i + 1]
    return th, tq

## 1. The opening: the 0.5 rad turn with three delays (Note, Section 1)

In [2]:
t = np.arange(0, 12.0005, 1e-3)
for tau in [0.0, 0.1, 0.25]:
    th, tq = simulate(t, 0.5, tau)
    pk = th[find_peaks(th)[0]]
    sat = t[(np.abs(tq) >= TAU_MAX - 1e-9) & (t > 1)]
    print(f"delay {tau:4.2f} s: peaks {np.round(pk[:3], 3)}, overshoot {100 * (pk[0] / 0.5 - 1):.0f} %,",
          f"torque at its limit after 1 s from {sat[0]:.2f} s" if len(sat) else "torque never at its limit after 1 s")

delay 0.00 s: peaks [0.763 0.573 0.52 ], overshoot 53 %, torque never at its limit after 1 s


delay 0.10 s: peaks [0.865 0.695 0.604], overshoot 73 %, torque never at its limit after 1 s
delay 0.25 s: peaks [1.128 1.478 1.949], overshoot 126 %, torque at its limit after 1 s from 1.25 s


## 2. The loop transfer function and the closed-loop poles (Note, Section 2)

In [3]:
poles = np.roots([1, 0.8, 4])          # s^2 + 0.8 s + 4 = 0
print("closed-loop poles", np.round(poles, 3))
print("L at a pole:", np.round(L(poles[0]), 12))
wgc = brentq(lambda w: abs(L(1j * w)) - 1, 0.1, 10)
print(f"gain crossover {wgc:.4f} rad/s, omega^2 = {wgc**2:.3f}, L there: length {abs(L(1j * wgc)):.3f},",
      f"angle {np.degrees(np.angle(L(1j * wgc))):.1f} deg")
pm = np.pi + np.angle(L(1j * wgc))
print(f"phase margin {np.degrees(pm):.1f} deg = {pm:.4f} rad; delay that makes L = -1: {pm / wgc:.4f} s")
print("check: L(j wgc) with that delay =", np.round(L(1j * wgc, pm / wgc), 6))

closed-loop poles [-0.4+1.96j -0.4-1.96j]
L at a pole: (-1-0j)
gain crossover 2.0815 rad/s, omega^2 = 4.333, L there: length 1.000, angle -157.4 deg
phase margin 22.6 deg = 0.3945 rad; delay that makes L = -1: 0.1895 s
check: L(j wgc) with that delay = (-1-0j)


## 3. Mapping one point and the argument principle (Note, Section 3)

In [4]:
s = -1 + 1j
print("s^2 =", s**2, " 0.8 s + 4 =", 0.8 * s + 4, " L(s) =", L(s))
print(f"arrow from the zero -5: length {abs(s + 5):.3f}, angle {np.degrees(np.angle(s + 5)):.1f} deg")
print(f"arrow from the pole 0: length {abs(s):.3f}, angle {np.degrees(np.angle(s)):.1f} deg")
print(f"length 0.8 x {abs(s + 5):.3f} / {abs(s):.3f}^2 = {0.8 * abs(s + 5) / abs(s)**2:.3f}; |L(s)| = {abs(L(s)):.3f}")
print(f"angle {np.degrees(np.angle(s + 5)):.1f} - 2 x {np.degrees(np.angle(s)):.1f} = "
      f"{np.degrees(np.angle(s + 5)) - 2 * np.degrees(np.angle(s)):.1f} deg; angle of L(s) {np.degrees(np.angle(L(s))):.1f} deg")


def cw_turns(z, about=0j):
    a = np.unwrap(np.angle(z - about))
    return -(a[-1] - a[0]) / (2 * np.pi)


f = lambda s: 1 + L(s)
z1 = -0.4 + 1j * np.sqrt(3.84)
phi = np.linspace(0, 2 * np.pi, 20001)
for c, r, name in [(z1, 0.6, "round the zero -0.4 + 1.96j"), (0j, 1.0, "round the double pole at 0"),
                   (1.6, 0.6, "round nothing"), (0j, 3.0, "round both zeros and the double pole")]:
    print(f"{name}: image of 1 + L turns {cw_turns(f(c + r * np.exp(-1j * phi))):+.0f} times clockwise round 0")

s^2 = -2j  0.8 s + 4 = (3.2+0.8j)  L(s) = (-0.39999999999999997+1.5999999999999999j)
arrow from the zero -5: length 4.123, angle 14.0 deg
arrow from the pole 0: length 1.414, angle 135.0 deg
length 0.8 x 4.123 / 1.414^2 = 1.649; |L(s)| = 1.649
angle 14.0 - 2 x 135.0 = -256.0 deg; angle of L(s) 104.0 deg
round the zero -0.4 + 1.96j: image of 1 + L turns +1 times clockwise round 0
round the double pole at 0: image of 1 + L turns -2 times clockwise round 0
round nothing: image of 1 + L turns +0 times clockwise round 0
round both zeros and the double pole: image of 1 + L turns +0 times clockwise round 0


## 4. The Nyquist count (Note, Section 4)

In [5]:
def contour(eps=1e-3, R=1e4, n=20000, small=True):
    w = np.logspace(np.log10(eps), np.log10(R), n)
    parts = [("omega > 0", 1j * w), ("huge half circle", R * np.exp(1j * np.linspace(np.pi / 2, -np.pi / 2, n))),
             ("omega < 0", -1j * w[::-1])]
    if small:
        parts.append(("tiny half circle", eps * np.exp(1j * np.linspace(-np.pi / 2, np.pi / 2, n))))
    return parts


for tau in [0.1, 0.25]:
    parts = contour()
    z = np.concatenate([L(p, tau) for _, p in parts])
    each = [cw_turns(np.concatenate([[L(parts[i - 1][1][-1], tau)] if i else [], L(p, tau)]), -1)
            for i, (_, p) in enumerate(parts)]
    print(f"delay {tau} s: clockwise turns round -1 by part {np.round(each, 2)}, total N = {cw_turns(z, -1):+.2f}")
    char = lambda s: s**2 + (0.8 * s + 4) * np.exp(-s * tau)
    root = fsolve(lambda x: [char(x[0] + 1j * x[1]).real, char(x[0] + 1j * x[1]).imag], [0.0, 2.0])
    print(f"   closed-loop pole near 2 rad/s: {root[0]:.3f} + {root[1]:.3f}j")

delay 0.1 s: clockwise turns round -1 by part [-0.5 -0.  -0.5  1. ], total N = -0.00


   closed-loop pole near 2 rad/s: -0.205 + 2.054j
delay 0.25 s: clockwise turns round -1 by part [ 0.5 -0.   0.5  1. ], total N = +2.00
   closed-loop pole near 2 rad/s: 0.139 + 2.064j


## 5. The balancing robot: a Bode reading misleads (Note, Section 5)

In [6]:
Lb = lambda s: (s + 2) / (s**2 - 1)
print("L_b(0) =", Lb(0), " poles of L_b:", np.roots([1, 0, -1]))
print("closed-loop poles, s^2 + s + 1 = 0:", np.round(np.roots([1, 1, 1]), 3))
zb = np.concatenate([Lb(p) for _, p in contour(small=False)])
print(f"clockwise turns round -1: N = {cw_turns(zb, -1):+.2f}")

L_b(0) = -2.0  poles of L_b: [-1.  1.]
closed-loop poles, s^2 + s + 1 = 0: [-0.5+0.866j -0.5-0.866j]
clockwise turns round -1: N = -1.00


## 6. Margins (Note, Section 6)

In [7]:
def margins(Lf):
    wgc = brentq(lambda w: abs(Lf(1j * w)) - 1, 1e-2, 1e2)
    pm = np.pi + np.angle(Lf(1j * wgc))
    w = np.logspace(-2, 3, 400001)
    ph = np.unwrap(np.angle(Lf(1j * w)))
    cross = np.where(np.diff(np.sign(ph + np.pi)) != 0)[0]
    wpc = w[cross[0]] if len(cross) else np.inf
    gm = 1 / abs(Lf(1j * wpc)) if len(cross) else np.inf
    return wgc, np.degrees(pm), pm / wgc, wpc, gm


for name, Lf in [("no delay", lambda s: L(s)), ("0.1 s delay", lambda s: L(s, 0.1))]:
    wgc, pm, dm, wpc, gm = margins(Lf)
    print(f"{name}: w_gc {wgc:.3f} rad/s, PM {pm:.1f} deg, DM {dm:.4f} s, w_pc {wpc:.2f} rad/s,",
          f"GM {gm:.2f} ({20 * np.log10(gm):.1f} dB), |L(j w_pc)| = {1 / gm:.4f}" if np.isfinite(gm) else "GM infinite")
# RO-060's tuned PD heading gains, Kp = 0.2, Kd = 0.283, at the delays of Figure 1 and Autoware's 0.17 s
Lt = lambda s, tau=0.0: (0.2 + 0.283 * s) / (I * s**2) * np.exp(-s * tau)
want = {0.0: (2.91, 76.4, np.inf, 0.458), 0.1: (2.91, 59.7, 5.38, 0.358), 0.17: (2.91, 48.0, 3.09, 0.288),
        0.25: (2.91, 34.6, 2.03, 0.208)}
for tau, (w0, pm0, gm0, dm0) in want.items():
    wgc, pm, dm, wpc, gm = margins(lambda s: Lt(s, tau))
    print(f"tuned PD, delay {tau} s: w_gc {wgc:.2f} rad/s, PM {pm:.1f} deg, GM {gm:.2f} at {wpc:.2f} rad/s, DM {dm:.3f} s")
    assert round(wgc, 2) == w0 and round(pm, 1) == pm0 and round(dm, 3) == dm0
    assert (np.isinf(gm) and np.isinf(gm0)) or round(gm, 2) == gm0
print(f"tuned PD noise gain |C(100j)| = {abs(0.2 + 0.283 * 100j):.1f} N m/rad")
L060 = lambda s: (0.283 * s**2 + 0.2 * s + 0.05) / (I * s**3)        # RO-060's tuned PID heading gains
wgc = brentq(lambda w: abs(L060(1j * w)) - 1, 0.5, 20)
pm = np.pi + np.angle(L060(1j * wgc))
print(f"RO-060's tuned gains (Kp 0.2, Kd 0.283, Ki 0.05): w_gc {wgc:.2f} rad/s, PM {np.degrees(pm):.1f} deg, DM {pm / wgc:.3f} s")
assert round(wgc, 2) == 2.86 and round(np.degrees(pm), 1) == 75.8 and round(pm / wgc, 3) == 0.463
print("phase of L (no delay) at 1, 10, 100 rad/s:", [round(np.degrees(np.angle(L(1j * w))), 1) for w in [1, 10, 100]])
print("any gain k: closed loop s^2 + 0.8 k s + 4 k; k = 13.4 roots", np.round(np.roots([1, 0.8 * 13.4, 4 * 13.4]), 2))

no delay: w_gc 2.082 rad/s, PM 22.6 deg, DM 0.1895 s, w_pc inf rad/s, GM infinite
0.1 s delay: w_gc 2.082 rad/s, PM 10.7 deg, DM 0.0895 s, w_pc 11.66 rad/s, GM 13.39 (22.5 dB), |L(j w_pc)| = 0.0747
tuned PD, delay 0.0 s: w_gc 2.91 rad/s, PM 76.4 deg, GM inf at inf rad/s, DM 0.458 s


tuned PD, delay 0.1 s: w_gc 2.91 rad/s, PM 59.7 deg, GM 5.38 at 15.24 rad/s, DM 0.358 s
tuned PD, delay 0.17 s: w_gc 2.91 rad/s, PM 48.0 deg, GM 3.09 at 8.77 rad/s, DM 0.288 s
tuned PD, delay 0.25 s: w_gc 2.91 rad/s, PM 34.6 deg, GM 2.03 at 5.80 rad/s, DM 0.208 s
tuned PD noise gain |C(100j)| = 28.3 N m/rad
RO-060's tuned gains (Kp 0.2, Kd 0.283, Ki 0.05): w_gc 2.86 rad/s, PM 75.8 deg, DM 0.463 s
phase of L (no delay) at 1, 10, 100 rad/s: [np.float64(-168.7), np.float64(-116.6), np.float64(-92.9)]
any gain k: closed loop s^2 + 0.8 k s + 4 k; k = 13.4 roots [-5.36+4.99j -5.36-4.99j]


## 7. Real stacks (Note, Section 7)

In [8]:
dm = margins(lambda s: L(s))[2]
print(f"Autoware's delay_compensation_time 0.17 s is {100 * 0.17 / dm:.0f} % of our delay margin {dm:.2f} s")
dmt = margins(lambda s: Lt(s))[2]
print(f"and {100 * 0.17 / dmt:.0f} % of the tuned loop's {dmt:.3f} s; tuned phase margin with 0.17 s: "
      f"{margins(lambda s: Lt(s, 0.17))[1]:.1f} deg")
assert round(100 * 0.17 / dm) == 90 and round(100 * 0.17 / dmt) == 37
for cut, d in [(30, 8e-3), (60, 3.8e-3), (120, 1.9e-3)]:
    print(f"PX4 gyro filter {cut} Hz, about {1000 * d:.1f} ms: our phase margin loses {np.degrees(2.0815 * d):.2f} deg")

Autoware's delay_compensation_time 0.17 s is 90 % of our delay margin 0.19 s


and 37 % of the tuned loop's 0.458 s; tuned phase margin with 0.17 s: 48.0 deg
PX4 gyro filter 30 Hz, about 8.0 ms: our phase margin loses 0.95 deg
PX4 gyro filter 60 Hz, about 3.8 ms: our phase margin loses 0.45 deg
PX4 gyro filter 120 Hz, about 1.9 ms: our phase margin loses 0.23 deg
